## 🧮 Feast로 재미있게 기능(Feature) 다루기

기계학습에서 모델은 잘 정의된 **기능(feature)**—예측 결과를 도출하는 데 도움이 되는 구조화된 데이터 포인트—에 의존합니다. 하지만 여러 프로젝트와 환경에서 이러한 기능들을 관리하는 것은 빠르게 복잡해질 수 있습니다. 학습(training)과 추론(inference) 간의 일관성을 어떻게 보장할까요? 팀 간에 기능들을 버전관리하고 공유하려면 어떻게 해야 할까요?

**Feature Store**가 바로 여기서 나옵니다. Feature Store는 머신러닝 기능을 저장, 처리 및 제공하는 **중앙집중식 저장소**로 작동합니다. 이를 통해 기능들이 학습과 실시간 추론 모두에서 일관성 있고 재사용 가능하며 효율적으로 검색될 수 있도록 합니다.

### 🔍 Feast란?

**Feast (Feature Store)**는 기능 관리를 간소화하는 오픈소스 프레임워크입니다. 이는 머신러닝 모델을 위한 기능을 저장, 검색 및 제공하는 **확장 가능하고 구조화된 방식**을 제공합니다. Feast를 사용함으로써 조직은:
1. 기능 정의 및 메타데이터를 **관리**할 수 있습니다.
2. 오프라인 및 온라인 데이터베이스에 기능을 **저장**할 수 있습니다.
3. 실시간 예측을 위해 기능을 빠르게 **제공**할 수 있습니다.

이 노트북에서 우리는 다음을 탐색할 것입니다:
1. **Feast**를 설정하고 기능 집합을 정의합니다.
2. **오프라인**에서 **온라인 저장소**로 기능을 물리화(materialize)합니다.
3. **머신러닝 워크플로우**를 위해 기능을 효율적으로 관리하는 방법을 이해합니다.

## ⚙️ Feast 설정하기

이제 Feast가 무엇인지 이해했으니, 이를 설정하고 우리의 **Feature Store**를 구성하고 상호작용하는 방법을 탐색해야 할 차례입니다.

기능 정의와 설정에 깊이 빠지기 전에, 먼저 **필요한 라이브러리와 의존성을 설치하고 가져와야** 합니다. 시작해봅시다!

In [ ]:
!pip install -q -r requirements.txt

In [2]:
import feast
from datetime import datetime
import yaml
import sys, os

### Feature Store 설정 파일

`feature_repo` 디렉토리 내에는 설정 및 기능 정의를 포함하는 여러 파일이 있습니다.

`features.py`에서 우리는 우리의 **Feature Store**에서 사용될 노래 기능들, 예를 들어 `energy`, `acousticness` 등의 목록을 정의합니다.

반면, Feast는 `feature_store.yaml`을 사용하여 **Feature Store**를 설정합니다. 이 파일은 우리의 경우 `feature_repo` 디렉토리인 **기능 저장소**의 루트에 위치해야 합니다.

In [9]:
sys.path.append(os.path.abspath('feature_repo/'))
from features import music, song_properties
from feature_service import song_properties_fs

In [ ]:
with open('feature_repo/feature_store.yaml', 'r') as file:
    fs_config_yaml = yaml.safe_load(file)

fs_config = feast.repo_config.RepoConfig(**fs_config_yaml)
fs = feast.FeatureStore(config=fs_config)

### 🏗️ Feast는 어떻게 작동할까요?

Feast는 기능들을 세 가지 핵심 구성요소로 정리합니다:
- **📜 Registry (레지스트리):** 모든 기능 정의, 소스 및 엔티티를 추적하는 메타데이터 저장소입니다.
- **📂 Offline Store (오프라인 저장소):** 모델 학습을 위한 역사적 기능 데이터를 보관하는 장기 저장 시스템입니다.
- **⚡ Online Store (온라인 저장소):** 추론 중 실시간 기능 검색에 최적화된 낮은 지연 시간의 저장소입니다.

In [ ]:
import yaml
# Pretty-print the YAML configuration
print(yaml.dump(fs_config_yaml, default_flow_style=False, sort_keys=False, indent=2))

앞서 언급했듯이, Feast는 `feature_store.yaml` 파일을 사용하여 [Feature Store를 설정합니다](https://docs.feast.dev/reference/feature-repository/feature-store-yaml#overview). YAML 설정을 살펴보면 우리가 방금 설명한 세 가지 핵심 구성요소를 인식할 수 있을 것입니다.

우리의 설정에서:
- **Registry**와 **Online Store**는 **PostgreSQL 데이터베이스**를 사용하도록 설정되어 있습니다.
- **Offline Store**는 **파일 기반 저장 시스템**으로 설정되어 있으며, S3 버킷과 연결될 수 있습니다(`feature.py`에서 S3 버킷을 FileStore로 사용하는 music_source를 확인하세요).

이 설정은 기능들이 학습과 추론 모두에 대해 효율적으로 저장, 추적 및 제공될 수 있도록 보장합니다.

## 🎯 기능(Features)과 기능값(Feature Values)

기계학습에서 **기능(feature)**은 예측 모델을 위한 입력 신호로 사용되는 핵심 데이터 조각입니다. 데이터셋의 맥락에서 우리는 다음을 구별합니다:

- **기능(Feature):** 측정 가능한 특성을 나타내는 데이터셋의 전체 열(예: 노래의 "에너지" 또는 "음성성")입니다.
- **기능값(Feature Value):** 해당 기능 열의 단일 데이터 포인트(예: 특정 노래의 "에너지" 값)입니다.

간단히 말해, 기능들은 모델이 예측을 하기 위해 사용하는 구조화된 정보를 제공합니다. 우리의 경우, 노래의 **에너지**와 **음성성**은 한 곡이 히트곡이 될지를 결정하는 데 도움이 될 수 있는 기능의 예입니다.

## 🚀 Feast 적용하기

기능을 사용하기 전에, 먼저 이들을 **적용(apply)**해야 합니다. 이 단계는 `feature_repo` 내의 모든 기능 정의를 우리의 **Feast 레지스트리**에 등록합니다.

우리의 설정에서 레지스트리는 중앙 메타데이터 저장소로 작동하는 **PostgreSQL 데이터베이스**에 저장됩니다. Feast를 적용함으로써, 모든 기능 정의가 적절히 카탈로그되고 오프라인 저장소에서 가져오거나 온라인 저장소에서 실시간으로 제공될 준비가 되도록 합니다.

In [5]:
fs.apply([song_properties_fs, music, song_properties])

## 기능 물리화(Materialize)하기
다음 단계는 기능을 물리화하는 것입니다. 이것은 기능을 오프라인 저장소에서 온라인 저장소로 이동합니다.
구체적으로, 정의된 시간범위 내에 있는 기능들의 부분집합을 이동하며, 온라인 저장소 내에는 최신 기능들만 저장합니다.

In [ ]:
fs.materialize(start_date=datetime(2023, 1, 1), end_date=datetime.now())

## 다음 단계
이제 Feast를 설정했으니, 사용을 시작해봅시다!
다음 노트북으로 이동하여 학습 기능을 가져오는 방법을 확인해봅시다: [2-test_load_historical_features.ipynb](2-test_load_historical_features.ipynb)